In [ ]:
# ======================================================================================
# TARGETED FINAL ECG RE-ANNOTATION TOOL
#
# PURPOSE
# -------
# Re-annotate ONLY the 14 leads whose V6 overlays showed crop-boundary clipping.
#
# Uses:
#     manual_digitization_qc/manual_geometry.json
#     original source ECG images
#
# Creates:
#     ecg_anchor_annotator_targeted.html
#
# Browser export:
#     ecg_anchor_annotations_targeted.json
#
# IMPORTANT
# ---------
# This does NOT alter X_gaze_final.npy.
# It only creates the new manual annotations.
#
# After annotation we will patch ONLY these 14 leads into the existing V6 array.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
import base64
import webbrowser

from pathlib import Path
from io import BytesIO

from PIL import Image


# ======================================================================================
# 1. PATHS
# ======================================================================================

GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)


if not GEOMETRY_PATH.exists():

    matches = list(
        Path(".").glob(
            "**/manual_geometry.json"
        )
    )

    if not matches:

        raise FileNotFoundError(
            "Could not find manual_geometry.json"
        )

    GEOMETRY_PATH = matches[0]


print(
    "Using geometry:",
    GEOMETRY_PATH
)


with open(
    GEOMETRY_PATH,
    "r"
) as f:

    geometry = json.load(f)


OUTPUT_HTML = Path(
    "ecg_anchor_annotator_targeted.html"
)


# ======================================================================================
# 2. ONLY THE 14 LEADS STILL NEEDING CORRECTION
# ======================================================================================

TARGET_LEADS = {

    "Hyperkalemia": [
        "V3",
    ],

    "VTach": [
        "II",
        "III",
        "aVF",
        "V2",
    ],

    "VentPaced": [
        "V2",
        "V3",
        "V4",
        "V5",
    ],

    "LBBB": [
        "V1",
        "V2",
        "V3",
    ],

    "CompleteHeartBlock": [
        "V1",
        "V2",
    ],
}


# ======================================================================================
# 3. DISPLAY GRID
# ======================================================================================

DISPLAY_GRID = [

    [
        "I",
        "aVR",
        "V1",
        "V4"
    ],

    [
        "II",
        "aVL",
        "V2",
        "V5"
    ],

    [
        "III",
        "aVF",
        "V3",
        "V6"
    ],
]


DISPLAY_POSITION = {}


for row_i, row in enumerate(
    DISPLAY_GRID
):

    for col_i, lead in enumerate(
        row
    ):

        DISPLAY_POSITION[
            lead
        ] = (
            row_i,
            col_i
        )


# ======================================================================================
# 4. EXPANDED VERTICAL CROP SETTINGS
#
# V4 used 0.62.
#
# These intentionally use much taller crops because these particular
# diagnoses have very large QRS excursions.
#
# It is acceptable if an adjacent lead becomes visible:
# YOU manually identify the intended lead.
# ======================================================================================

HALF_HEIGHT_SCALE = {

    "Hyperkalemia":
        0.90,

    "VTach":
        1.00,

    "VentPaced":
        0.95,

    "LBBB":
        1.05,

    "CompleteHeartBlock":
        0.95,
}


# Keep essentially the same horizontal lead interval as V4.
X_PAD_FRACTION = 0.025


# Browser requirements
MIN_POINTS = 10

MAX_FIRST_X_FRACTION = 0.10
MIN_LAST_X_FRACTION = 0.90


# ======================================================================================
# 5. SOURCE IMAGE HELPERS
# ======================================================================================

def resolve_source_image(
    stored_path
):

    p = Path(
        stored_path
    )


    if p.exists():

        return p


    matches = list(
        Path(".").glob(
            f"**/{p.name}"
        )
    )


    if not matches:

        raise FileNotFoundError(
            f"Could not find source image: {p.name}"
        )


    return matches[0]


# ======================================================================================
# 6. GENERATE LARGE TARGET-LEAD CROP
# ======================================================================================

def get_expanded_crop_box(
    ecg,
    lead
):

    g = geometry[
        ecg
    ]


    rows = [
        float(x)
        for x
        in g[
            "row_centres"
        ]
    ]


    xs = [
        float(x)
        for x
        in g[
            "x_boundaries"
        ]
    ]


    H = int(
        g[
            "image_height"
        ]
    )


    W = int(
        g[
            "image_width"
        ]
    )


    row_i, col_i = (
        DISPLAY_POSITION[
            lead
        ]
    )


    # ----------------------------------------------------------------------------------
    # Local diagnostic-row spacing
    # ----------------------------------------------------------------------------------

    if row_i == 0:

        spacing = (
            rows[
                1
            ]
            -
            rows[
                0
            ]
        )


    elif row_i == 2:

        spacing = (
            rows[
                2
            ]
            -
            rows[
                1
            ]
        )


    else:

        spacing = min(

            rows[
                1
            ]
            -
            rows[
                0
            ],

            rows[
                2
            ]
            -
            rows[
                1
            ],
        )


    half_height = (
        HALF_HEIGHT_SCALE[
            ecg
        ]
        *
        spacing
    )


    centre_y = rows[
        row_i
    ]


    y0 = int(
        round(
            centre_y
            -
            half_height
        )
    )


    y1 = int(
        round(
            centre_y
            +
            half_height
        )
    )


    y0 = max(
        0,
        y0
    )


    y1 = min(
        H,
        y1
    )


    # ----------------------------------------------------------------------------------
    # Standard four-column boundaries
    # ----------------------------------------------------------------------------------

    x0 = xs[
        col_i
    ]


    x1 = xs[
        col_i
        +
        1
    ]


    width = (
        x1
        -
        x0
    )


    x0 = int(
        round(
            x0
            +
            X_PAD_FRACTION
            *
            width
        )
    )


    x1 = int(
        round(
            x1
            -
            X_PAD_FRACTION
            *
            width
        )
    )


    # ----------------------------------------------------------------------------------
    # Target row-centre position within expanded crop.
    # Useful as a visual orientation guide.
    # ----------------------------------------------------------------------------------

    target_centre_y = (
        centre_y
        -
        y0
    )


    return {

        "box": [
            x0,
            y0,
            x1,
            y1
        ],

        "target_centre_y":
            float(
                target_centre_y
            ),

        "source_width":
            W,

        "source_height":
            H,
    }


# ======================================================================================
# 7. IMAGE -> BASE64
# ======================================================================================

def image_to_data_uri(
    image
):

    buffer = BytesIO()


    image.save(
        buffer,
        format="PNG"
    )


    encoded = (
        base64
        .b64encode(
            buffer.getvalue()
        )
        .decode(
            "ascii"
        )
    )


    return (
        "data:image/png;base64,"
        +
        encoded
    )


# ======================================================================================
# 8. BUILD TASKS
# ======================================================================================

tasks = []


for ecg, leads in TARGET_LEADS.items():

    source_path = resolve_source_image(
        geometry[
            ecg
        ][
            "image_path"
        ]
    )


    source = (
        Image.open(
            source_path
        )
        .convert(
            "RGB"
        )
    )


    for lead in leads:

        info = get_expanded_crop_box(
            ecg,
            lead
        )


        x0, y0, x1, y1 = (
            info[
                "box"
            ]
        )


        crop = source.crop(
            (
                x0,
                y0,
                x1,
                y1
            )
        )


        original_w = (
            x1
            -
            x0
        )


        original_h = (
            y1
            -
            y0
        )


        # --------------------------------------------------------------------------------
        # Upscale for easier browser clicking
        # --------------------------------------------------------------------------------

        target_display_width = 1500


        if (
            crop.width
            <
            target_display_width
        ):

            scale = (
                target_display_width
                /
                crop.width
            )


            crop = crop.resize(

                (
                    int(
                        round(
                            crop.width
                            *
                            scale
                        )
                    ),

                    int(
                        round(
                            crop.height
                            *
                            scale
                        )
                    ),
                ),

                Image.Resampling.LANCZOS
            )


        display_scale_y = (
            crop.height
            /
            original_h
        )


        target_centre_display_y = (
            info[
                "target_centre_y"
            ]
            *
            display_scale_y
        )


        tasks.append(
            {

                "ecg":
                    ecg,

                "lead":
                    lead,

                "original_box":
                    [
                        x0,
                        y0,
                        x1,
                        y1
                    ],

                "original_crop_width":
                    original_w,

                "original_crop_height":
                    original_h,

                "display_width":
                    crop.width,

                "display_height":
                    crop.height,

                "target_centre_display_y":
                    float(
                        target_centre_display_y
                    ),

                "half_height_scale":
                    HALF_HEIGHT_SCALE[
                        ecg
                    ],

                "full_horizontal_span":
                    True,

                "image":
                    image_to_data_uri(
                        crop
                    ),
            }
        )


print(
    "\nNumber of targeted tasks:",
    len(
        tasks
    )
)


assert len(
    tasks
) == 14


# ======================================================================================
# 9. HTML APP
# ======================================================================================

tasks_json = json.dumps(
    tasks
)


html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>Final targeted ECG annotation</title>

<style>

body {

    font-family:
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        Arial,
        sans-serif;

    margin: 0;

    background:
        #f4f4f4;

    color:
        #222;
}


header {

    position:
        sticky;

    top:
        0;

    z-index:
        100;

    background:
        white;

    border-bottom:
        1px solid #ccc;

    padding:
        12px 20px;
}


h1 {

    margin:
        0 0 5px 0;

    font-size:
        22px;
}


#progress {

    font-weight:
        600;
}


main {

    max-width:
        1700px;

    margin:
        auto;

    padding:
        18px;
}


.instructions {

    background:
        white;

    padding:
        16px 20px;

    border-radius:
        10px;

    margin-bottom:
        16px;

    line-height:
        1.55;
}


.warning {

    background:
        #fff4d8;

    padding:
        12px 16px;

    border-radius:
        8px;

    margin-top:
        10px;
}


#leadTitle {

    font-size:
        27px;

    font-weight:
        700;

    margin:
        12px 0;
}


#canvasWrapper {

    width:
        100%;

    overflow-x:
        auto;

    background:
        white;

    border-radius:
        10px;

    padding:
        10px;

    box-sizing:
        border-box;
}


canvas {

    display:
        block;

    cursor:
        crosshair;

    border:
        1px solid #777;

    max-width:
        none;
}


.controls {

    display:
        flex;

    flex-wrap:
        wrap;

    gap:
        8px;

    margin-top:
        14px;
}


button {

    border:
        1px solid #999;

    border-radius:
        7px;

    background:
        white;

    padding:
        9px 14px;

    font-size:
        14px;

    cursor:
        pointer;
}


button:hover {

    background:
        #eee;
}


.primary {

    background:
        #222;

    color:
        white;

    border-color:
        #222;
}


.accept {

    background:
        #e5f3ff;
}


#status {

    margin-top:
        12px;

    padding:
        10px 14px;

    border-radius:
        7px;

    background:
        #eee;
}


#coverage {

    margin-left:
        12px;

    font-weight:
        600;
}

</style>

</head>


<body>


<header>

<h1>
Final targeted ECG anchor annotation
</h1>

<div id="progress"></div>

</header>


<main>


<div class="instructions">

<strong>These are ONLY the 14 leads that were still clipped in V6.</strong>

<br><br>

The crop is deliberately much taller than before.
You may therefore see parts of neighbouring ECG rows.

<strong>
Ignore neighbouring traces and annotate ONLY the lead named at the top.
</strong>

<br><br>

Click along the true black ECG waveform from
<strong>LEFT → RIGHT</strong>.

For each complex, include enough points to preserve:

baseline → QRS → ST segment → T wave → next complex.

Use plenty of points around large VT/LBBB/paced complexes.

<div class="warning">

<strong>Important horizontal rule:</strong>

Your first point must be near the LEFT edge and your final point
must be near the RIGHT edge.

The tool will not allow you to save a correction unless your
anchors span almost the entire diagnostic lead.

This prevents partial ECG intervals being stretched to 1000 samples.

</div>

</div>


<div id="leadTitle"></div>


<div id="canvasWrapper">

<canvas id="canvas"></canvas>

</div>


<div class="controls">

<button onclick="previousTask()">
← Previous
</button>

<button onclick="undoPoint()">
Undo
</button>

<button onclick="clearPoints()">
Clear
</button>

<button
    class="accept"
    onclick="saveCorrected()"
>
Save corrected
</button>

<button onclick="nextTask()">
Next →
</button>

<button
    class="primary"
    onclick="exportJSON()"
>
Export JSON
</button>

<span id="coverage"></span>

</div>


<div id="status"></div>


</main>


<script>


const tasks =
    __TASKS__;


const STORAGE_KEY =
    "ecg_targeted_annotations_v2";


const MIN_POINTS =
    __MIN_POINTS__;


const MAX_FIRST_X_FRACTION =
    __MAX_FIRST__;


const MIN_LAST_X_FRACTION =
    __MIN_LAST__;


let annotations = {};


try {

    const stored =
        localStorage.getItem(
            STORAGE_KEY
        );


    if (stored) {

        annotations =
            JSON.parse(
                stored
            );

    }

} catch(err) {

    console.log(err);

}


let currentIndex =
    0;


const canvas =
    document.getElementById(
        "canvas"
    );


const ctx =
    canvas.getContext(
        "2d"
    );


const image =
    new Image();


function taskKey(
    task
) {

    return (
        task.ecg
        +
        "::"
        +
        task.lead
    );

}


function getAnnotation(
    task
) {

    const key =
        taskKey(
            task
        );


    if (
        !annotations[
            key
        ]
    ) {

        annotations[
            key
        ] = {

            ecg:
                task.ecg,

            lead:
                task.lead,

            status:
                "pending",

            points:
                [],

            original_box:
                task.original_box,

            original_crop_width:
                task.original_crop_width,

            original_crop_height:
                task.original_crop_height,

            display_width:
                task.display_width,

            display_height:
                task.display_height,

            half_height_scale:
                task.half_height_scale,

            full_horizontal_span:
                true
        };

    }


    return annotations[
        key
    ];

}


function saveLocal() {

    localStorage.setItem(

        STORAGE_KEY,

        JSON.stringify(
            annotations
        )
    );

}


function renderTask() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        getAnnotation(
            task
        );


    document.getElementById(
        "leadTitle"
    ).innerText =
        task.ecg
        +
        " — "
        +
        task.lead;


    image.onload =
        function() {


            canvas.width =
                image.naturalWidth;


            canvas.height =
                image.naturalHeight;


            draw();

        };


    image.src =
        task.image;


    updateStatus();
    updateProgress();

}


function draw() {

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    ctx.drawImage(
        image,
        0,
        0,
        canvas.width,
        canvas.height
    );


    const task =
        tasks[
            currentIndex
        ];


    const ann =
        getAnnotation(
            task
        );


    // -----------------------------------------------------------------
    // Faint guide showing manually clicked diagnostic-row centre.
    //
    // This is ONLY orientation.
    // The actual ECG may deviate substantially above/below this.
    // -----------------------------------------------------------------

    ctx.save();

    ctx.setLineDash(
        [
            8,
            8
        ]
    );

    ctx.strokeStyle =
        "rgba(0,150,255,0.55)";

    ctx.lineWidth =
        2;

    ctx.beginPath();

    ctx.moveTo(
        0,
        task.target_centre_display_y
    );

    ctx.lineTo(
        canvas.width,
        task.target_centre_display_y
    );

    ctx.stroke();

    ctx.restore();


    const points =
        ann.points;


    if (
        points.length
        ===
        0
    ) {

        return;

    }


    // -----------------------------------------------------------------
    // Yellow manual trajectory
    // -----------------------------------------------------------------

    ctx.strokeStyle =
        "#ffd400";

    ctx.lineWidth =
        3;

    ctx.beginPath();


    points.forEach(
        (p, i) => {


            if (
                i
                ===
                0
            ) {

                ctx.moveTo(
                    p.x,
                    p.y
                );

            } else {

                ctx.lineTo(
                    p.x,
                    p.y
                );

            }

        }
    );


    ctx.stroke();


    // -----------------------------------------------------------------
    // Anchor markers
    // -----------------------------------------------------------------

    points.forEach(
        (p, i) => {


            ctx.beginPath();


            ctx.arc(
                p.x,
                p.y,
                5,
                0,
                2 * Math.PI
            );


            ctx.fillStyle =
                "#ff2929";


            ctx.fill();


            ctx.strokeStyle =
                "white";


            ctx.lineWidth =
                1;


            ctx.stroke();


            ctx.fillStyle =
                "black";


            ctx.font =
                "11px Arial";


            ctx.fillText(
                String(
                    i + 1
                ),
                p.x + 6,
                p.y - 6
            );

        }
    );

}


canvas.addEventListener(

    "click",

    function(event) {


        const rect =
            canvas.getBoundingClientRect();


        const scaleX =
            canvas.width
            /
            rect.width;


        const scaleY =
            canvas.height
            /
            rect.height;


        const x =
            (
                event.clientX
                -
                rect.left
            )
            *
            scaleX;


        const y =
            (
                event.clientY
                -
                rect.top
            )
            *
            scaleY;


        const task =
            tasks[
                currentIndex
            ];


        const ann =
            getAnnotation(
                task
            );


        // Enforce strict left-to-right order
        if (
            ann.points.length
            >
            0
            &&
            x
            <=
            ann.points[
                ann.points.length - 1
            ].x
        ) {

            alert(
                "Please annotate strictly from LEFT to RIGHT.\n"
                +
                "Use Undo if you need to go backwards."
            );

            return;

        }


        ann.points.push(
            {
                x:
                    x,

                y:
                    y
            }
        );


        ann.status =
            "editing";


        saveLocal();

        draw();

        updateStatus();

    }
);


function undoPoint() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        getAnnotation(
            task
        );


    if (
        ann.points.length
        >
        0
    ) {

        ann.points.pop();


        ann.status =
            "editing";


        saveLocal();

        draw();

        updateStatus();

    }

}


function clearPoints() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        getAnnotation(
            task
        );


    if (
        confirm(
            "Clear all points for this lead?"
        )
    ) {

        ann.points =
            [];


        ann.status =
            "pending";


        saveLocal();

        draw();

        updateStatus();

    }

}


function saveCorrected() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        getAnnotation(
            task
        );


    if (
        ann.points.length
        <
        MIN_POINTS
    ) {

        alert(
            "Please place at least "
            +
            MIN_POINTS
            +
            " points."
        );

        return;

    }


    const first =
        ann.points[
            0
        ];


    const last =
        ann.points[
            ann.points.length - 1
        ];


    const firstFraction =
        first.x
        /
        task.display_width;


    const lastFraction =
        last.x
        /
        task.display_width;


    if (
        firstFraction
        >
        MAX_FIRST_X_FRACTION
    ) {

        alert(
            "Your FIRST anchor is too far from the left edge.\n\n"
            +
            "Add/redo anchors so the annotation begins within "
            +
            Math.round(
                MAX_FIRST_X_FRACTION * 100
            )
            +
            "% of the left edge."
        );

        return;

    }


    if (
        lastFraction
        <
        MIN_LAST_X_FRACTION
    ) {

        alert(
            "Your FINAL anchor is too far from the right edge.\n\n"
            +
            "Continue annotating until at least "
            +
            Math.round(
                MIN_LAST_X_FRACTION * 100
            )
            +
            "% across the crop."
        );

        return;

    }


    ann.status =
        "corrected";


    ann.horizontal_coverage = {

        first_fraction:
            firstFraction,

        last_fraction:
            lastFraction,

        span_fraction:
            (
                last.x
                -
                first.x
            )
            /
            task.display_width
    };


    saveLocal();

    updateStatus();

    updateProgress();


    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        currentIndex += 1;

        renderTask();

    }

}


function nextTask() {

    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        currentIndex += 1;

        renderTask();

    }

}


function previousTask() {

    if (
        currentIndex
        >
        0
    ) {

        currentIndex -= 1;

        renderTask();

    }

}


function updateStatus() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        getAnnotation(
            task
        );


    const statusElement =
        document.getElementById(
            "status"
        );


    statusElement.innerText =
        "Status: "
        +
        ann.status;


    if (
        ann.status
        ===
        "corrected"
    ) {

        statusElement.style.background =
            "#e5f3ff";

    } else {

        statusElement.style.background =
            "#eeeeee";

    }


    let coverageText =
        ann.points.length
        +
        " anchors";


    if (
        ann.points.length
        >
        1
    ) {

        const first =
            ann.points[
                0
            ];


        const last =
            ann.points[
                ann.points.length - 1
            ];


        const coverage =
            (
                last.x
                -
                first.x
            )
            /
            task.display_width;


        coverageText +=
            " | horizontal coverage: "
            +
            Math.round(
                coverage
                *
                100
            )
            +
            "%";

    }


    document.getElementById(
        "coverage"
    ).innerText =
        coverageText;

}


function updateProgress() {

    let completed =
        0;


    tasks.forEach(
        task => {


            const ann =
                getAnnotation(
                    task
                );


            if (
                ann.status
                ===
                "corrected"
            ) {

                completed += 1;

            }

        }
    );


    document.getElementById(
        "progress"
    ).innerText =
        "Lead "
        +
        (
            currentIndex
            +
            1
        )
        +
        " / "
        +
        tasks.length
        +
        " | completed "
        +
        completed
        +
        " / "
        +
        tasks.length;

}


function exportJSON() {

    saveLocal();


    let incomplete =
        [];


    tasks.forEach(
        task => {


            const ann =
                getAnnotation(
                    task
                );


            if (
                ann.status
                !==
                "corrected"
            ) {

                incomplete.push(
                    task.ecg
                    +
                    " "
                    +
                    task.lead
                );

            }

        }
    );


    if (
        incomplete.length
        >
        0
    ) {

        const proceed =
            confirm(
                "The following leads are not marked corrected:\n\n"
                +
                incomplete.join(
                    "\n"
                )
                +
                "\n\nExport anyway?"
            );


        if (
            !proceed
        ) {

            return;

        }

    }


    const result = {

        format:
            "ECG_targeted_anchor_annotations_v2",

        purpose:
            "Expanded-crop correction of 14 clipped V6 leads",

        task_count:
            tasks.length,

        full_horizontal_reconstruction_required:
            true,

        annotations:
            annotations
    };


    const blob =
        new Blob(

            [
                JSON.stringify(
                    result,
                    null,
                    2
                )
            ],

            {
                type:
                    "application/json"
            }
        );


    const url =
        URL.createObjectURL(
            blob
        );


    const a =
        document.createElement(
            "a"
        );


    a.href =
        url;


    a.download =
        "ecg_anchor_annotations_targeted.json";


    document.body.appendChild(
        a
    );


    a.click();


    document.body.removeChild(
        a
    );


    URL.revokeObjectURL(
        url
    );

}


renderTask();


</script>


</body>

</html>
"""


html = html.replace(
    "__TASKS__",
    tasks_json
)


html = html.replace(
    "__MIN_POINTS__",
    str(
        MIN_POINTS
    )
)


html = html.replace(
    "__MAX_FIRST__",
    str(
        MAX_FIRST_X_FRACTION
    )
)


html = html.replace(
    "__MIN_LAST__",
    str(
        MIN_LAST_X_FRACTION
    )
)


# ======================================================================================
# 10. SAVE HTML
# ======================================================================================

with open(
    OUTPUT_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html
    )


print(
    "\n" +
    "=" * 100
)

print(
    "TARGETED ANNOTATOR CREATED"
)

print(
    "=" * 100
)


print(
    "\nHTML:"
)

print(
    OUTPUT_HTML.resolve()
)


print(
    "\nTasks:"
)


for i, task in enumerate(
    tasks,
    start=1
):

    print(
        f"{i:02d}. "
        f"{task['ecg']:22s} "
        f"{task['lead']}"
    )


print(
    """
HOW TO USE IT

Open ecg_anchor_annotator_targeted.html in Chrome or Safari.

For each of the 14 leads:

    • identify ONLY the named lead
    • ignore neighbouring traces visible in the taller crop
    • click from LEFT → RIGHT
    • cover essentially the ENTIRE width
    • use >=10 anchors
    • use many more anchors around large QRS complexes
    • click "Save corrected"

The faint blue horizontal line is only an orientation guide.
DO NOT trace the blue line — trace the actual black ECG.

When all 14 are complete:

    click "Export JSON"

It will download:

    ecg_anchor_annotations_targeted.json

Leave that file in LOCAL_USER_HOME/Downloads/.
"""
)


# Try opening automatically
try:

    webbrowser.open(
        OUTPUT_HTML.resolve().as_uri()
    )

except Exception:

    pass

In [ ]:
# ======================================================================================
# FINAL LBBB 3-LEAD ULTRA-TALL ANNOTATOR
#
# PURPOSE
# -------
# Re-annotate ONLY:
#
#       LBBB V1
#       LBBB V2
#       LBBB V3
#
# These were the only remaining leads with genuine repeated crop-boundary clipping
# after V7 QC.
#
# The crop is expanded MUCH MORE BELOW the diagnostic row than above it because
# the unresolved clipping is at the lower edge of the large negative QRS complexes.
#
# OUTPUT:
#       ecg_anchor_annotator_LBBB_final.html
#
# Browser export:
#       ecg_anchor_annotations_LBBB_final.json
#
# This does NOT modify the ECG dataset yet.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
import base64
import webbrowser

from pathlib import Path
from io import BytesIO

from PIL import Image


# ======================================================================================
# 1. PATHS
# ======================================================================================

GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)


if not GEOMETRY_PATH.exists():

    matches = list(
        Path(".").glob(
            "**/manual_geometry.json"
        )
    )

    if not matches:

        raise FileNotFoundError(
            "Could not find manual_geometry.json"
        )

    GEOMETRY_PATH = matches[0]


print(
    "Using geometry:",
    GEOMETRY_PATH
)


with open(
    GEOMETRY_PATH,
    "r"
) as f:

    geometry = json.load(f)


OUTPUT_HTML = Path(
    "ecg_anchor_annotator_LBBB_final.html"
)


# ======================================================================================
# 2. ONLY THREE REMAINING LEADS
# ======================================================================================

ECG = "LBBB"

TARGET_LEADS = [
    "V1",
    "V2",
    "V3",
]


# ======================================================================================
# 3. DISPLAY GRID
# ======================================================================================

DISPLAY_GRID = [

    ["I",   "aVR", "V1", "V4"],

    ["II",  "aVL", "V2", "V5"],

    ["III", "aVF", "V3", "V6"],
]


DISPLAY_POSITION = {}


for r, row in enumerate(
    DISPLAY_GRID
):

    for c, lead in enumerate(
        row
    ):

        DISPLAY_POSITION[
            lead
        ] = (
            r,
            c
        )


# ======================================================================================
# 4. FINAL CROP SETTINGS
#
# Previous symmetric LBBB crop:
#       ~1.05 × row spacing
#
# Remaining clipping:
#       LOWER border only.
#
# Therefore use an asymmetric crop:
#
#       upward  = 0.80 × row spacing
#       downward = 1.65 × row spacing
#
# This gives substantial additional room below the negative QRS while avoiding
# unnecessary expansion above the intended diagnostic lead.
# ======================================================================================

UPWARD_SCALE = 0.80
DOWNWARD_SCALE = 1.65

X_PAD_FRACTION = 0.025

MIN_POINTS = 14

MAX_FIRST_X_FRACTION = 0.10
MIN_LAST_X_FRACTION = 0.90


# ======================================================================================
# 5. SOURCE IMAGE
# ======================================================================================

def resolve_source_image(
    stored_path
):

    path = Path(
        stored_path
    )


    if path.exists():

        return path


    matches = list(
        Path(".").glob(
            f"**/{path.name}"
        )
    )


    if not matches:

        raise FileNotFoundError(
            f"Could not locate source ECG image: {path.name}"
        )


    return matches[0]


source_path = resolve_source_image(
    geometry[
        ECG
    ][
        "image_path"
    ]
)


print(
    "LBBB source ECG:",
    source_path
)


source = (
    Image.open(
        source_path
    )
    .convert(
        "RGB"
    )
)


# ======================================================================================
# 6. CREATE ASYMMETRIC EXPANDED CROP
# ======================================================================================

def get_final_crop_box(
    lead
):

    g = geometry[
        ECG
    ]


    rows = [
        float(v)
        for v
        in g[
            "row_centres"
        ]
    ]


    xs = [
        float(v)
        for v
        in g[
            "x_boundaries"
        ]
    ]


    H = int(
        g[
            "image_height"
        ]
    )


    W = int(
        g[
            "image_width"
        ]
    )


    row_i, col_i = (
        DISPLAY_POSITION[
            lead
        ]
    )


    # ----------------------------------------------------------------------------------
    # Local row spacing
    # ----------------------------------------------------------------------------------

    if row_i == 0:

        spacing = (
            rows[1]
            -
            rows[0]
        )

    elif row_i == 2:

        spacing = (
            rows[2]
            -
            rows[1]
        )

    else:

        spacing = min(
            rows[1]
            -
            rows[0],

            rows[2]
            -
            rows[1],
        )


    centre_y = rows[
        row_i
    ]


    # ----------------------------------------------------------------------------------
    # ASYMMETRIC crop
    # ----------------------------------------------------------------------------------

    y0 = int(
        round(
            centre_y
            -
            UPWARD_SCALE
            *
            spacing
        )
    )


    y1 = int(
        round(
            centre_y
            +
            DOWNWARD_SCALE
            *
            spacing
        )
    )


    y0 = max(
        0,
        y0
    )


    y1 = min(
        H,
        y1
    )


    # ----------------------------------------------------------------------------------
    # Horizontal diagnostic-lead boundaries
    # ----------------------------------------------------------------------------------

    x0 = xs[
        col_i
    ]


    x1 = xs[
        col_i + 1
    ]


    width = (
        x1
        -
        x0
    )


    x0 = int(
        round(
            x0
            +
            X_PAD_FRACTION
            *
            width
        )
    )


    x1 = int(
        round(
            x1
            -
            X_PAD_FRACTION
            *
            width
        )
    )


    target_centre_y = (
        centre_y
        -
        y0
    )


    return {

        "box": [
            x0,
            y0,
            x1,
            y1
        ],

        "target_centre_y":
            float(
                target_centre_y
            ),

        "row_spacing":
            float(
                spacing
            ),
    }


# ======================================================================================
# 7. IMAGE -> EMBEDDED DATA URI
# ======================================================================================

def image_to_data_uri(
    image
):

    buffer = BytesIO()


    image.save(
        buffer,
        format="PNG"
    )


    encoded = (
        base64
        .b64encode(
            buffer.getvalue()
        )
        .decode(
            "ascii"
        )
    )


    return (
        "data:image/png;base64,"
        +
        encoded
    )


# ======================================================================================
# 8. BUILD THREE TASKS
# ======================================================================================

tasks = []


for lead in TARGET_LEADS:

    info = get_final_crop_box(
        lead
    )


    x0, y0, x1, y1 = (
        info[
            "box"
        ]
    )


    crop = source.crop(
        (
            x0,
            y0,
            x1,
            y1
        )
    )


    original_width = (
        x1
        -
        x0
    )


    original_height = (
        y1
        -
        y0
    )


    # ----------------------------------------------------------------------------------
    # Upscale browser view
    # ----------------------------------------------------------------------------------

    target_width = 1600


    if crop.width < target_width:

        scale = (
            target_width
            /
            crop.width
        )


        crop = crop.resize(

            (
                int(
                    round(
                        crop.width
                        *
                        scale
                    )
                ),

                int(
                    round(
                        crop.height
                        *
                        scale
                    )
                ),
            ),

            Image.Resampling.LANCZOS
        )


    display_scale_y = (
        crop.height
        /
        original_height
    )


    target_centre_display_y = (
        info[
            "target_centre_y"
        ]
        *
        display_scale_y
    )


    tasks.append({

        "ecg":
            ECG,

        "lead":
            lead,

        "original_box":
            [
                x0,
                y0,
                x1,
                y1
            ],

        "original_crop_width":
            original_width,

        "original_crop_height":
            original_height,

        "display_width":
            crop.width,

        "display_height":
            crop.height,

        "target_centre_display_y":
            float(
                target_centre_display_y
            ),

        "upward_scale":
            UPWARD_SCALE,

        "downward_scale":
            DOWNWARD_SCALE,

        "image":
            image_to_data_uri(
                crop
            ),
    })


print(
    "\nFinal annotation tasks:"
)


for task in tasks:

    print(
        task[
            "ecg"
        ],
        task[
            "lead"
        ],
        "box =",
        task[
            "original_box"
        ]
    )


assert len(tasks) == 3


# ======================================================================================
# 9. CREATE HTML ANNOTATOR
# ======================================================================================

tasks_json = json.dumps(
    tasks
)


html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>LBBB final 3-lead annotation</title>

<style>

body {

    font-family:
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        Arial,
        sans-serif;

    margin: 0;

    background: #f4f4f4;

    color: #222;
}


header {

    position: sticky;

    top: 0;

    z-index: 100;

    background: white;

    border-bottom: 1px solid #ccc;

    padding: 12px 20px;
}


h1 {

    margin: 0 0 5px 0;

    font-size: 23px;
}


#progress {

    font-weight: 600;
}


main {

    max-width: 1800px;

    margin: auto;

    padding: 18px;
}


.instructions {

    background: white;

    padding: 17px 20px;

    border-radius: 10px;

    line-height: 1.55;

    margin-bottom: 16px;
}


.warning {

    background: #fff1d6;

    border-radius: 8px;

    padding: 12px 15px;

    margin-top: 12px;
}


#leadTitle {

    font-size: 30px;

    font-weight: 700;

    margin: 12px 0;
}


#canvasWrapper {

    width: 100%;

    overflow-x: auto;

    background: white;

    padding: 10px;

    border-radius: 10px;

    box-sizing: border-box;
}


canvas {

    display: block;

    max-width: none;

    cursor: crosshair;

    border: 1px solid #777;
}


.controls {

    display: flex;

    gap: 8px;

    flex-wrap: wrap;

    margin-top: 14px;
}


button {

    border: 1px solid #999;

    border-radius: 7px;

    padding: 9px 15px;

    background: white;

    cursor: pointer;

    font-size: 14px;
}


button:hover {

    background: #eee;
}


.accept {

    background: #e5f3ff;
}


.primary {

    background: #222;

    color: white;

    border-color: #222;
}


#status {

    margin-top: 12px;

    padding: 10px 14px;

    background: #eee;

    border-radius: 7px;
}


#coverage {

    margin-left: 10px;

    font-weight: 600;
}

</style>

</head>


<body>


<header>

<h1>
LBBB — final V1/V2/V3 annotation
</h1>

<div id="progress"></div>

</header>


<main>


<div class="instructions">

Only three leads remain:

<strong>V1, V2 and V3.</strong>

<br><br>

These crops have been expanded substantially
<strong>below the diagnostic row</strong>
because that is where the previous QRS complexes were clipped.

You will see neighbouring ECG traces.

<strong>
Ignore them completely.
Trace only the lead named at the top.
</strong>

<br><br>

Follow the intended black ECG waveform from
<strong>LEFT → RIGHT</strong>.

Use plenty of anchors around the very steep downward QRS excursion,
particularly around:

baseline → onset → steep descent → deepest trough → steep ascent →
ST/T morphology.

<div class="warning">

<strong>Critical:</strong>

The deepest negative QRS should now remain fully visible.

If the intended waveform STILL physically leaves the image,
do not improvise its shape.

Instead leave that lead unsaved and tell me.

</div>

</div>


<div id="leadTitle"></div>


<div id="canvasWrapper">

<canvas id="canvas"></canvas>

</div>


<div class="controls">

<button onclick="previousTask()">
← Previous
</button>

<button onclick="undoPoint()">
Undo
</button>

<button onclick="clearPoints()">
Clear
</button>

<button
    class="accept"
    onclick="saveCorrected()"
>
Save corrected
</button>

<button onclick="nextTask()">
Next →
</button>

<button
    class="primary"
    onclick="exportJSON()"
>
Export JSON
</button>

<span id="coverage"></span>

</div>


<div id="status"></div>


</main>


<script>


const tasks =
    __TASKS__;


const MIN_POINTS =
    __MIN_POINTS__;


const STORAGE_KEY =
    "LBBB_final_three_leads_v1";


let annotations = {};


try {

    const stored =
        localStorage.getItem(
            STORAGE_KEY
        );


    if (stored) {

        annotations =
            JSON.parse(
                stored
            );

    }

} catch(err) {

    console.log(err);

}


let currentIndex = 0;


const canvas =
    document.getElementById(
        "canvas"
    );


const ctx =
    canvas.getContext(
        "2d"
    );


const image =
    new Image();


function keyFor(
    task
) {

    return (
        task.ecg
        +
        "::"
        +
        task.lead
    );

}


function annotationFor(
    task
) {

    const key =
        keyFor(
            task
        );


    if (!annotations[key]) {

        annotations[key] = {

            ecg:
                task.ecg,

            lead:
                task.lead,

            status:
                "pending",

            points:
                [],

            original_box:
                task.original_box,

            original_crop_width:
                task.original_crop_width,

            original_crop_height:
                task.original_crop_height,

            display_width:
                task.display_width,

            display_height:
                task.display_height,

            upward_scale:
                task.upward_scale,

            downward_scale:
                task.downward_scale
        };

    }


    return annotations[key];

}


function saveLocal() {

    localStorage.setItem(

        STORAGE_KEY,

        JSON.stringify(
            annotations
        )
    );

}


function renderTask() {

    const task =
        tasks[
            currentIndex
        ];


    document.getElementById(
        "leadTitle"
    ).innerText =
        task.ecg
        +
        " — "
        +
        task.lead;


    image.onload =
        function() {


            canvas.width =
                image.naturalWidth;


            canvas.height =
                image.naturalHeight;


            draw();

        };


    image.src =
        task.image;


    updateStatus();
    updateProgress();

}


function draw() {

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    ctx.drawImage(
        image,
        0,
        0,
        canvas.width,
        canvas.height
    );


    const task =
        tasks[
            currentIndex
        ];


    const ann =
        annotationFor(
            task
        );


    // -----------------------------------------------------------------
    // Blue row-centre orientation guide.
    // DO NOT TRACE THIS LINE.
    // -----------------------------------------------------------------

    ctx.save();

    ctx.setLineDash(
        [8, 8]
    );

    ctx.strokeStyle =
        "rgba(0,150,255,0.45)";

    ctx.lineWidth =
        2;

    ctx.beginPath();

    ctx.moveTo(
        0,
        task.target_centre_display_y
    );

    ctx.lineTo(
        canvas.width,
        task.target_centre_display_y
    );

    ctx.stroke();

    ctx.restore();


    const points =
        ann.points;


    if (points.length === 0) {

        return;

    }


    // Yellow anchor trajectory
    ctx.strokeStyle =
        "#ffd400";

    ctx.lineWidth =
        3;

    ctx.beginPath();


    points.forEach(
        (p, i) => {


            if (i === 0) {

                ctx.moveTo(
                    p.x,
                    p.y
                );

            } else {

                ctx.lineTo(
                    p.x,
                    p.y
                );

            }

        }
    );


    ctx.stroke();


    // Red anchor points
    points.forEach(
        (p, i) => {


            ctx.beginPath();


            ctx.arc(
                p.x,
                p.y,
                5,
                0,
                Math.PI * 2
            );


            ctx.fillStyle =
                "#ff2929";


            ctx.fill();


            ctx.strokeStyle =
                "white";


            ctx.lineWidth =
                1;


            ctx.stroke();


            ctx.font =
                "11px Arial";


            ctx.fillStyle =
                "black";


            ctx.fillText(
                String(
                    i + 1
                ),
                p.x + 6,
                p.y - 6
            );

        }
    );

}


canvas.addEventListener(

    "click",

    function(event) {


        const rect =
            canvas.getBoundingClientRect();


        const scaleX =
            canvas.width
            /
            rect.width;


        const scaleY =
            canvas.height
            /
            rect.height;


        const x =
            (
                event.clientX
                -
                rect.left
            )
            *
            scaleX;


        const y =
            (
                event.clientY
                -
                rect.top
            )
            *
            scaleY;


        const task =
            tasks[
                currentIndex
            ];


        const ann =
            annotationFor(
                task
            );


        if (
            ann.points.length > 0
            &&
            x <= ann.points[
                ann.points.length - 1
            ].x
        ) {

            alert(
                "Please annotate strictly from LEFT to RIGHT."
            );

            return;

        }


        ann.points.push(
            {
                x: x,
                y: y
            }
        );


        ann.status =
            "editing";


        saveLocal();

        draw();

        updateStatus();

    }
);


function undoPoint() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        annotationFor(
            task
        );


    if (ann.points.length > 0) {

        ann.points.pop();

        ann.status =
            "editing";

        saveLocal();

        draw();

        updateStatus();

    }

}


function clearPoints() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        annotationFor(
            task
        );


    if (
        confirm(
            "Clear all points for this lead?"
        )
    ) {

        ann.points = [];

        ann.status =
            "pending";

        saveLocal();

        draw();

        updateStatus();

    }

}


function saveCorrected() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        annotationFor(
            task
        );


    if (
        ann.points.length
        <
        MIN_POINTS
    ) {

        alert(
            "Please place at least "
            +
            MIN_POINTS
            +
            " anchors."
        );

        return;

    }


    const first =
        ann.points[0];


    const last =
        ann.points[
            ann.points.length - 1
        ];


    const firstFraction =
        first.x
        /
        task.display_width;


    const lastFraction =
        last.x
        /
        task.display_width;


    if (
        firstFraction
        >
        0.10
    ) {

        alert(
            "Your first anchor must be within 10% of the LEFT edge."
        );

        return;

    }


    if (
        lastFraction
        <
        0.90
    ) {

        alert(
            "Your last anchor must reach at least 90% across the lead."
        );

        return;

    }


    ann.status =
        "corrected";


    ann.horizontal_coverage = {

        first_fraction:
            firstFraction,

        last_fraction:
            lastFraction,

        span_fraction:
            (
                last.x
                -
                first.x
            )
            /
            task.display_width
    };


    saveLocal();

    updateStatus();
    updateProgress();


    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        currentIndex += 1;

        renderTask();

    }

}


function nextTask() {

    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        currentIndex += 1;

        renderTask();

    }

}


function previousTask() {

    if (
        currentIndex
        >
        0
    ) {

        currentIndex -= 1;

        renderTask();

    }

}


function updateStatus() {

    const task =
        tasks[
            currentIndex
        ];


    const ann =
        annotationFor(
            task
        );


    document.getElementById(
        "status"
    ).innerText =
        "Status: "
        +
        ann.status;


    let text =
        ann.points.length
        +
        " anchors";


    if (
        ann.points.length
        >
        1
    ) {

        const first =
            ann.points[0];


        const last =
            ann.points[
                ann.points.length - 1
            ];


        const coverage =
            (
                last.x
                -
                first.x
            )
            /
            task.display_width;


        text +=
            " | coverage: "
            +
            Math.round(
                coverage * 100
            )
            +
            "%";

    }


    document.getElementById(
        "coverage"
    ).innerText =
        text;

}


function updateProgress() {

    let complete = 0;


    tasks.forEach(
        task => {


            if (
                annotationFor(
                    task
                ).status
                ===
                "corrected"
            ) {

                complete += 1;

            }

        }
    );


    document.getElementById(
        "progress"
    ).innerText =
        "Lead "
        +
        (currentIndex + 1)
        +
        " / 3"
        +
        " | completed "
        +
        complete
        +
        " / 3";

}


function exportJSON() {

    saveLocal();


    const unfinished = [];


    tasks.forEach(
        task => {


            if (
                annotationFor(
                    task
                ).status
                !==
                "corrected"
            ) {

                unfinished.push(
                    task.lead
                );

            }

        }
    );


    if (
        unfinished.length
        >
        0
    ) {

        alert(
            "Please finish all three leads before export:\n\n"
            +
            unfinished.join(
                "\n"
            )
        );

        return;

    }


    const result = {

        format:
            "LBBB_final_three_lead_annotations_v1",

        purpose:
            "Final correction of residual lower-edge clipping",

        task_count:
            3,

        annotations:
            annotations
    };


    const blob =
        new Blob(

            [
                JSON.stringify(
                    result,
                    null,
                    2
                )
            ],

            {
                type:
                    "application/json"
            }
        );


    const url =
        URL.createObjectURL(
            blob
        );


    const a =
        document.createElement(
            "a"
        );


    a.href =
        url;


    a.download =
        "ecg_anchor_annotations_LBBB_final.json";


    document.body.appendChild(
        a
    );


    a.click();


    document.body.removeChild(
        a
    );


    URL.revokeObjectURL(
        url
    );

}


renderTask();


</script>


</body>

</html>
"""


html = html.replace(
    "__TASKS__",
    tasks_json
)


html = html.replace(
    "__MIN_POINTS__",
    str(
        MIN_POINTS
    )
)


# ======================================================================================
# 10. SAVE AND OPEN
# ======================================================================================

with open(
    OUTPUT_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html
    )


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL THREE-LEAD LBBB ANNOTATOR CREATED"
)

print(
    "=" * 100
)


print(
    "\nOpen:"
)

print(
    OUTPUT_HTML.resolve()
)


print(
    """
Annotate only:

    LBBB V1
    LBBB V2
    LBBB V3

The black neighbouring traces are irrelevant.

For each intended lead:
    • cover the whole width
    • use >=14 points
    • use several points down and back up the deep QRS
    • confirm the deepest trough is physically visible
    • Save corrected

Then click Export JSON.

Expected download:

    ecg_anchor_annotations_LBBB_final.json

Leave it in LOCAL_USER_HOME/Downloads/

After that, tell me "done".
"""
)


try:

    webbrowser.open(
        OUTPUT_HTML.resolve().as_uri()
    )

except Exception:

    pass

In [ ]:
# ======================================================================================
# ABSOLUTE FINAL ANNOTATION — LBBB V2 ONLY
#
# PURPOSE
# -------
# Re-annotate ONLY LBBB V2 using a much deeper downward crop.
#
# Previous V8:
#     minimum distance to bottom edge = 0 px
#     bottom-edge contact = 1.55%
#
# This version expands much farther DOWNWARD so the full negative QRS should
# remain visible with a genuine margin beneath it.
#
# OUTPUT:
#     ecg_anchor_annotator_LBBB_V2_final.html
#
# Browser export:
#     ecg_anchor_annotations_LBBB_V2_final.json
#
# This code DOES NOT modify X_gaze_final.npy.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
import base64
import webbrowser

from pathlib import Path
from io import BytesIO

from PIL import Image


# ======================================================================================
# 1. PATHS
# ======================================================================================

GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)

if not GEOMETRY_PATH.exists():

    matches = list(
        Path(".").glob(
            "**/manual_geometry.json"
        )
    )

    if not matches:

        raise FileNotFoundError(
            "Could not find manual_geometry.json"
        )

    GEOMETRY_PATH = matches[0]


with open(
    GEOMETRY_PATH,
    "r"
) as f:

    geometry = json.load(f)


OUTPUT_HTML = Path(
    "ecg_anchor_annotator_LBBB_V2_final.html"
)


ECG = "LBBB"
LEAD = "V2"


print(
    "Geometry:",
    GEOMETRY_PATH
)


# ======================================================================================
# 2. DISPLAY POSITION
# ======================================================================================

DISPLAY_GRID = [

    ["I",   "aVR", "V1", "V4"],

    ["II",  "aVL", "V2", "V5"],

    ["III", "aVF", "V3", "V6"],
]


DISPLAY_POSITION = {}


for r, row in enumerate(
    DISPLAY_GRID
):

    for c, lead in enumerate(
        row
    ):

        DISPLAY_POSITION[
            lead
        ] = (
            r,
            c
        )


# ======================================================================================
# 3. ULTRA-DEEP CROP SETTINGS
#
# V8 used:
#     upward   = 0.80 × row spacing
#     downward = 1.65 × row spacing
#
# V2 still touched the lower boundary.
#
# Final:
#     upward   = 0.80
#     downward = 2.40
#
# We deliberately prefer too much surrounding image over any residual clipping.
# Neighbouring traces are fine because YOU manually identify V2.
# ======================================================================================

UPWARD_SCALE = 0.80

DOWNWARD_SCALE = 2.40

X_PAD_FRACTION = 0.025


# Require dense tracing because this is the definitive single-lead correction.
MIN_POINTS = 20

MAX_FIRST_X_FRACTION = 0.08
MIN_LAST_X_FRACTION = 0.92


# ======================================================================================
# 4. FIND SOURCE ECG
# ======================================================================================

def resolve_source_image(
    stored_path
):

    p = Path(
        stored_path
    )

    if p.exists():

        return p


    matches = list(
        Path(".").glob(
            f"**/{p.name}"
        )
    )


    if not matches:

        raise FileNotFoundError(
            f"Could not find source image: {p.name}"
        )


    return matches[0]


source_path = resolve_source_image(
    geometry[
        ECG
    ][
        "image_path"
    ]
)


source = (
    Image.open(
        source_path
    )
    .convert(
        "RGB"
    )
)


print(
    "Source ECG:",
    source_path
)


# ======================================================================================
# 5. COMPUTE FINAL V2 CROP
# ======================================================================================

g = geometry[
    ECG
]


rows = [
    float(v)
    for v
    in g[
        "row_centres"
    ]
]


xs = [
    float(v)
    for v
    in g[
        "x_boundaries"
    ]
]


H = int(
    g[
        "image_height"
    ]
)


W = int(
    g[
        "image_width"
    ]
)


row_i, col_i = (
    DISPLAY_POSITION[
        LEAD
    ]
)


# V2 is the middle diagnostic row.
spacing = min(

    rows[1]
    -
    rows[0],

    rows[2]
    -
    rows[1],
)


centre_y = rows[
    row_i
]


# --------------------------------------------------------------------------------------
# Much deeper asymmetric crop
# --------------------------------------------------------------------------------------

y0 = int(
    round(
        centre_y
        -
        UPWARD_SCALE
        *
        spacing
    )
)


y1 = int(
    round(
        centre_y
        +
        DOWNWARD_SCALE
        *
        spacing
    )
)


y0 = max(
    0,
    y0
)


y1 = min(
    H,
    y1
)


# --------------------------------------------------------------------------------------
# Standard diagnostic-column width
# --------------------------------------------------------------------------------------

x0 = xs[
    col_i
]


x1 = xs[
    col_i + 1
]


segment_width = (
    x1
    -
    x0
)


x0 = int(
    round(
        x0
        +
        X_PAD_FRACTION
        *
        segment_width
    )
)


x1 = int(
    round(
        x1
        -
        X_PAD_FRACTION
        *
        segment_width
    )
)


crop = source.crop(
    (
        x0,
        y0,
        x1,
        y1
    )
)


original_width = (
    x1
    -
    x0
)


original_height = (
    y1
    -
    y0
)


target_centre_y = (
    centre_y
    -
    y0
)


print(
    "\nFinal LBBB V2 crop:"
)

print(
    "x:",
    x0,
    "->",
    x1
)

print(
    "y:",
    y0,
    "->",
    y1
)

print(
    "original size:",
    original_width,
    "x",
    original_height
)

print(
    "row spacing:",
    spacing
)


# ======================================================================================
# 6. UPSCALE FOR BROWSER
# ======================================================================================

TARGET_DISPLAY_WIDTH = 1700


if crop.width < TARGET_DISPLAY_WIDTH:

    scale = (
        TARGET_DISPLAY_WIDTH
        /
        crop.width
    )


    crop = crop.resize(

        (
            int(
                round(
                    crop.width
                    *
                    scale
                )
            ),

            int(
                round(
                    crop.height
                    *
                    scale
                )
            ),
        ),

        Image.Resampling.LANCZOS
    )


display_scale_y = (
    crop.height
    /
    original_height
)


target_centre_display_y = (
    target_centre_y
    *
    display_scale_y
)


# ======================================================================================
# 7. EMBED IMAGE
# ======================================================================================

buffer = BytesIO()


crop.save(
    buffer,
    format="PNG"
)


encoded = (
    base64
    .b64encode(
        buffer.getvalue()
    )
    .decode(
        "ascii"
    )
)


image_uri = (
    "data:image/png;base64,"
    +
    encoded
)


task = {

    "ecg":
        ECG,

    "lead":
        LEAD,

    "original_box":
        [
            x0,
            y0,
            x1,
            y1
        ],

    "original_crop_width":
        original_width,

    "original_crop_height":
        original_height,

    "display_width":
        crop.width,

    "display_height":
        crop.height,

    "target_centre_display_y":
        float(
            target_centre_display_y
        ),

    "upward_scale":
        UPWARD_SCALE,

    "downward_scale":
        DOWNWARD_SCALE,

    "image":
        image_uri,
}


task_json = json.dumps(
    task
)


# ======================================================================================
# 8. HTML ANNOTATOR
# ======================================================================================

html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>FINAL LBBB V2 annotation</title>

<style>

body {

    font-family:
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        Arial,
        sans-serif;

    margin:
        0;

    background:
        #f4f4f4;

    color:
        #222;
}


header {

    background:
        white;

    border-bottom:
        1px solid #ccc;

    padding:
        14px 20px;
}


h1 {

    margin:
        0;

    font-size:
        24px;
}


main {

    max-width:
        1900px;

    margin:
        auto;

    padding:
        20px;
}


.instructions {

    background:
        white;

    padding:
        18px 22px;

    border-radius:
        10px;

    line-height:
        1.55;

    margin-bottom:
        16px;
}


.important {

    background:
        #fff0d3;

    padding:
        13px 16px;

    border-radius:
        8px;

    margin-top:
        12px;
}


#canvasWrapper {

    width:
        100%;

    overflow-x:
        auto;

    background:
        white;

    padding:
        10px;

    border-radius:
        10px;

    box-sizing:
        border-box;
}


canvas {

    display:
        block;

    max-width:
        none;

    cursor:
        crosshair;

    border:
        1px solid #777;
}


.controls {

    display:
        flex;

    flex-wrap:
        wrap;

    gap:
        8px;

    margin-top:
        15px;
}


button {

    padding:
        10px 16px;

    border:
        1px solid #999;

    border-radius:
        7px;

    background:
        white;

    cursor:
        pointer;

    font-size:
        14px;
}


button:hover {

    background:
        #eee;
}


.save {

    background:
        #e4f2ff;
}


.export {

    background:
        #222;

    color:
        white;

    border-color:
        #222;
}


#status {

    margin-top:
        12px;

    padding:
        10px 14px;

    border-radius:
        7px;

    background:
        #eee;

    font-weight:
        600;
}

</style>

</head>


<body>


<header>

<h1>
FINAL ECG correction — LBBB V2 only
</h1>

</header>


<main>


<div class="instructions">

This is the <strong>only remaining ECG lead</strong> requiring correction.

<br><br>

The image has intentionally been expanded much farther downward.

You will therefore see several neighbouring ECG traces.

<strong>
Ignore every neighbouring trace. Trace only diagnostic lead V2.
</strong>

<br><br>

Follow V2 from the far left to the far right.

Use multiple anchors around every major complex, especially:

baseline → beginning of descent → steep descending limb →
<strong>deepest QRS trough</strong> → ascending limb → ST/T wave.

<div class="important">

<strong>Before saving:</strong>

The bottom of BOTH large negative V2 QRS complexes must be completely visible
with actual white/grid space underneath them.

If the V2 waveform still disappears through the bottom edge,
DO NOT save the annotation.

</div>

</div>


<div id="canvasWrapper">

<canvas id="canvas"></canvas>

</div>


<div class="controls">

<button onclick="undoPoint()">
Undo
</button>

<button onclick="clearPoints()">
Clear
</button>

<button
    class="save"
    onclick="saveCorrected()"
>
Save corrected
</button>

<button
    class="export"
    onclick="exportJSON()"
>
Export JSON
</button>

</div>


<div id="status"></div>


</main>


<script>


const task =
    __TASK__;


const MIN_POINTS =
    __MIN_POINTS__;


const MAX_FIRST =
    __MAX_FIRST__;


const MIN_LAST =
    __MIN_LAST__;


const STORAGE_KEY =
    "LBBB_V2_absolute_final_v1";


let annotation = {

    ecg:
        task.ecg,

    lead:
        task.lead,

    status:
        "pending",

    points:
        [],

    original_box:
        task.original_box,

    original_crop_width:
        task.original_crop_width,

    original_crop_height:
        task.original_crop_height,

    display_width:
        task.display_width,

    display_height:
        task.display_height,

    upward_scale:
        task.upward_scale,

    downward_scale:
        task.downward_scale
};


try {

    const stored =
        localStorage.getItem(
            STORAGE_KEY
        );


    if (stored) {

        annotation =
            JSON.parse(
                stored
            );

    }

} catch(err) {

    console.log(err);

}


const canvas =
    document.getElementById(
        "canvas"
    );


const ctx =
    canvas.getContext(
        "2d"
    );


const image =
    new Image();


function saveLocal() {

    localStorage.setItem(
        STORAGE_KEY,
        JSON.stringify(
            annotation
        )
    );

}


function render() {

    image.onload =
        function() {


            canvas.width =
                image.naturalWidth;


            canvas.height =
                image.naturalHeight;


            draw();

        };


    image.src =
        task.image;


    updateStatus();

}


function draw() {

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    ctx.drawImage(
        image,
        0,
        0,
        canvas.width,
        canvas.height
    );


    // -----------------------------------------------------------------
    // Blue row centre = orientation only.
    // -----------------------------------------------------------------

    ctx.save();


    ctx.setLineDash(
        [8, 8]
    );


    ctx.strokeStyle =
        "rgba(0,150,255,0.40)";


    ctx.lineWidth =
        2;


    ctx.beginPath();


    ctx.moveTo(
        0,
        task.target_centre_display_y
    );


    ctx.lineTo(
        canvas.width,
        task.target_centre_display_y
    );


    ctx.stroke();


    ctx.restore();


    if (
        annotation.points.length
        ===
        0
    ) {

        return;

    }


    // -----------------------------------------------------------------
    // Yellow connecting path
    // -----------------------------------------------------------------

    ctx.strokeStyle =
        "#ffd400";


    ctx.lineWidth =
        3;


    ctx.beginPath();


    annotation.points.forEach(
        (p, i) => {


            if (
                i
                ===
                0
            ) {

                ctx.moveTo(
                    p.x,
                    p.y
                );

            } else {

                ctx.lineTo(
                    p.x,
                    p.y
                );

            }

        }
    );


    ctx.stroke();


    // -----------------------------------------------------------------
    // Red numbered anchors
    // -----------------------------------------------------------------

    annotation.points.forEach(
        (p, i) => {


            ctx.beginPath();


            ctx.arc(
                p.x,
                p.y,
                5,
                0,
                2 * Math.PI
            );


            ctx.fillStyle =
                "#ff2727";


            ctx.fill();


            ctx.strokeStyle =
                "white";


            ctx.lineWidth =
                1;


            ctx.stroke();


            ctx.font =
                "11px Arial";


            ctx.fillStyle =
                "black";


            ctx.fillText(
                String(
                    i + 1
                ),
                p.x + 6,
                p.y - 6
            );

        }
    );

}


canvas.addEventListener(

    "click",

    function(event) {


        const rect =
            canvas.getBoundingClientRect();


        const scaleX =
            canvas.width
            /
            rect.width;


        const scaleY =
            canvas.height
            /
            rect.height;


        const x =
            (
                event.clientX
                -
                rect.left
            )
            *
            scaleX;


        const y =
            (
                event.clientY
                -
                rect.top
            )
            *
            scaleY;


        if (
            annotation.points.length
            >
            0
            &&
            x
            <=
            annotation.points[
                annotation.points.length - 1
            ].x
        ) {

            alert(
                "Please annotate strictly LEFT → RIGHT."
            );

            return;

        }


        annotation.points.push(
            {
                x:
                    x,

                y:
                    y
            }
        );


        annotation.status =
            "editing";


        saveLocal();

        draw();

        updateStatus();

    }
);


function undoPoint() {

    if (
        annotation.points.length
        >
        0
    ) {

        annotation.points.pop();


        annotation.status =
            "editing";


        saveLocal();

        draw();

        updateStatus();

    }

}


function clearPoints() {

    if (
        confirm(
            "Clear all LBBB V2 anchors?"
        )
    ) {

        annotation.points =
            [];


        annotation.status =
            "pending";


        saveLocal();

        draw();

        updateStatus();

    }

}


function saveCorrected() {

    if (
        annotation.points.length
        <
        MIN_POINTS
    ) {

        alert(
            "Please place at least "
            +
            MIN_POINTS
            +
            " anchors."
        );

        return;

    }


    const first =
        annotation.points[
            0
        ];


    const last =
        annotation.points[
            annotation.points.length - 1
        ];


    const firstFraction =
        first.x
        /
        task.display_width;


    const lastFraction =
        last.x
        /
        task.display_width;


    if (
        firstFraction
        >
        MAX_FIRST
    ) {

        alert(
            "Start closer to the LEFT edge."
        );

        return;

    }


    if (
        lastFraction
        <
        MIN_LAST
    ) {

        alert(
            "Continue tracing closer to the RIGHT edge."
        );

        return;

    }


    annotation.status =
        "corrected";


    annotation.horizontal_coverage = {

        first_fraction:
            firstFraction,

        last_fraction:
            lastFraction,

        span_fraction:
            (
                last.x
                -
                first.x
            )
            /
            task.display_width
    };


    saveLocal();

    updateStatus();


    alert(
        "LBBB V2 saved as corrected.\n\n"
        +
        "Now click Export JSON."
    );

}


function updateStatus() {

    let text =
        "Status: "
        +
        annotation.status
        +
        " | "
        +
        annotation.points.length
        +
        " anchors";


    if (
        annotation.points.length
        >
        1
    ) {

        const first =
            annotation.points[
                0
            ];


        const last =
            annotation.points[
                annotation.points.length - 1
            ];


        const coverage =
            (
                last.x
                -
                first.x
            )
            /
            task.display_width;


        text +=
            " | coverage "
            +
            Math.round(
                100
                *
                coverage
            )
            +
            "%";

    }


    document.getElementById(
        "status"
    ).innerText =
        text;

}


function exportJSON() {

    if (
        annotation.status
        !==
        "corrected"
    ) {

        alert(
            "Save the annotation as corrected before exporting."
        );

        return;

    }


    const result = {

        format:
            "LBBB_V2_absolute_final_annotation_v1",

        purpose:
            "Final replacement of residual clipped LBBB V2",

        annotations: {

            "LBBB::V2":
                annotation
        }
    };


    const blob =
        new Blob(

            [
                JSON.stringify(
                    result,
                    null,
                    2
                )
            ],

            {
                type:
                    "application/json"
            }
        );


    const url =
        URL.createObjectURL(
            blob
        );


    const a =
        document.createElement(
            "a"
        );


    a.href =
        url;


    a.download =
        "ecg_anchor_annotations_LBBB_V2_final.json";


    document.body.appendChild(
        a
    );


    a.click();


    document.body.removeChild(
        a
    );


    URL.revokeObjectURL(
        url
    );

}


render();


</script>


</body>

</html>
"""


html = html.replace(
    "__TASK__",
    task_json
)


html = html.replace(
    "__MIN_POINTS__",
    str(
        MIN_POINTS
    )
)


html = html.replace(
    "__MAX_FIRST__",
    str(
        MAX_FIRST_X_FRACTION
    )
)


html = html.replace(
    "__MIN_LAST__",
    str(
        MIN_LAST_X_FRACTION
    )
)


# ======================================================================================
# 9. SAVE + OPEN
# ======================================================================================

with open(
    OUTPUT_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html
    )


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL LBBB V2 ANNOTATOR CREATED"
)

print(
    "=" * 100
)


print(
    "\nOpen:"
)

print(
    OUTPUT_HTML.resolve()
)


print(
    """
TRACE ONLY LBBB V2.

Before tracing, visually confirm that BOTH deep negative QRS troughs
are completely visible with grid/white space underneath them.

Then:

    • trace left -> right
    • use >=20 anchors
    • trace both QRS complexes densely
    • start within 8% of left edge
    • finish beyond 92% of the width
    • Save corrected
    • Export JSON

Expected download:

    ecg_anchor_annotations_LBBB_V2_final.json

Leave it in:

    LOCAL_USER_HOME/Downloads/

Then tell me: done
"""
)


try:

    webbrowser.open(
        OUTPUT_HTML.resolve().as_uri()
    )

except Exception:

    pass